In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')

df_Q1=pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:
df_Q1.head()

In [ ]:
# Task 3: Write your code here:
df_Q1.info()

In [ ]:
# Task 4: Write your code here:
df_Q1.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_Q1['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_Q1 = df_Q1.drop(columns='Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_Q1)

In [ ]:
# Jandling missing Values
for col in ['Delivery_Time', 'Courier_Experience_yrs']:
    df_Q1[col] = df_Q1[col].fillna(df_Q1[col].mean())

df_Q1 = df_Q1.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day'])

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_Q1)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_Q1.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_Q1[col] = le.fit_transform(df_Q1[col])
  label_encoders[col] = le

df_Q1

In [ ]:
# Task 5: Write your code here:
numerical_cols = df_Q1.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # WE DON'T SCALE THE TARGET

scaler = StandardScaler()
df_Q1[numerical_cols] = scaler.fit_transform(df_Q1[numerical_cols])
df_Q1.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_Q1, "Delivery_Time")
#class distributions are not equal, then our data is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df_Q1.drop("Delivery_Time", axis=1).astype(float)
y = df_Q1['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

mae_scores = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: